# data_entry_automatizado — cuaderno del curso

**Este cuaderno es para dar el curso, no para leerlo.** Cada sección tiene un
par de líneas de contexto y celdas que se ejecutan en vivo. La idea es tenerlo
en pantalla mientras hablás.

Corre igual en **Google Colab** y en local. En Colab clona el repo e instala
todo; en local, si ya estás parado en el repo, no hace nada.

---

### Cómo está organizado

| Sección | De qué va |
|---|---|
| 0 | Preparar el entorno |
| 1 | El problema |
| 2 | El contrato antes que el código |
| 3 | Datos falsos y ground truth |
| 4 | El dominio: reglas sucias |
| 5 | El extractor de PDF |
| 6 | Enchufar DOCX |
| 7 | Revisión humana: el Excel |
| 8 | La web falsa |
| 9 | La carga |
| 10 | Idempotencia y conflictos |
| 11 | Adaptarlo a lo tuyo |

> Todos los datos son inventados. La web de destino también viene en el repo.

## 0. Preparar el entorno

Una sola celda. En Colab tarda un par de minutos, casi todo bajando Chromium.

In [ ]:
# Funciona en Colab y en local. Si ya estás dentro del repo, no hace nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/GEJ1/data_entry_automatizado.git"

def corriendo_en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if not Path("pipeline/contratos.py").exists():
    if Path("data_entry_automatizado/pipeline/contratos.py").exists():
        os.chdir("data_entry_automatizado")
    else:
        print("Clonando el repo...")
        r = subprocess.run(["git", "clone", "-q", REPO], capture_output=True, text=True)
        if r.returncode != 0:
            raise SystemExit(
                "No se pudo clonar. Si el repo es privado, hacelo público o cloná\n"
                "a mano con un token:\n"
                "  !git clone https://<TOKEN>@github.com/GEJ1/data_entry_automatizado.git\n\n"
                + r.stderr)
        os.chdir("data_entry_automatizado")

print("carpeta de trabajo:", Path.cwd())

if corriendo_en_colab():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "playwright", "install", "--with-deps",
                    "chromium"], check=True)
    print("dependencias instaladas")
else:
    print("local: se asume que el venv ya está activado")

print("python", sys.version.split()[0], "(el proyecto necesita 3.10+)")

In [ ]:
# Ayudantes para mostrar cosas en el cuaderno.
import subprocess, sys, textwrap, time
from pathlib import Path
from IPython.display import Image, Markdown, display

PY = sys.executable

def correr(comando, mostrar=True):
    """Corre un comando del proyecto y muestra su salida."""
    r = subprocess.run(comando, shell=True, capture_output=True, text=True)
    salida = (r.stdout + r.stderr).rstrip()
    if mostrar:
        print(salida)
    return salida

def cli(args, mostrar=True):
    return correr(f"{PY} -m pipeline.cli {args}", mostrar)

def ver_codigo(ruta, desde=1, hasta=None):
    """Muestra un archivo del repo con números de línea."""
    lineas = Path(ruta).read_text(encoding="utf-8").splitlines()
    hasta = hasta or len(lineas)
    for i, l in enumerate(lineas[desde - 1:hasta], start=desde):
        print(f"{i:>4} | {l}")

print("listo")

## 1. El problema

Un reporte con fichas de clientes. Cada ficha tiene dos tablas, y alguien las
tiene que pasar a mano a un formulario web.

**La pregunta que abre el curso:** si el programa se equivoca, ¿cómo te enterás?
Un humano cansado se equivoca al azar; un programa comete el *mismo* error 300
veces.

In [ ]:
# El lote de demostración: 15 clientes, con once trampas sembradas a propósito.
cli("extraer --help", mostrar=False)
correr(f"{PY} demo/generar_pdf_fake.py")

In [ ]:
# El mismo generador, a escala. Esto es lo que hace inviable el trabajo manual.
correr(f"{PY} demo/generar_pdf_fake.py --clientes 300 "
       f"--salida /tmp/lote300.pdf | head -3")

import pdfplumber
with pdfplumber.open("/tmp/lote300.pdf") as pdf:
    print(f"\n{len(pdf.pages)} páginas, 300 clientes, dos tablas cada uno.")

## 2. El contrato antes que el código

Lo primero que se decide no es la librería: es **dónde están las junturas**.

`contratos.py` no hace nada, y ese es exactamente el punto. Define la forma que
tienen que tener las piezas para encajar.

La regla de oro: **el Extractor no sabe de negocio.** No sabe qué es un CUIT ni
cómo se lee una antigüedad. Por eso se puede reemplazar.

In [ ]:
ver_codigo("pipeline/contratos.py", 28, 75)

## 3. Datos falsos y ground truth

La sección más importante del curso, y la que casi nadie hace.

El generador **sabe exactamente qué escribió** en el archivo, así que puede
dejar al lado el resultado esperado. Comparar contra eso es la diferencia entre
*"parece que anda"* y *"anda"*.

In [ ]:
# Las once trampas, cada una con nombre y razón de ser.
ver_codigo("demo/datos_fake.py", 148, 168)

In [ ]:
# El ground truth: lo que un extractor correcto TIENE que devolver.
import json
verdad = json.loads(Path("data/entrada/lote17.verdad.json").read_text(encoding="utf-8"))
print("fichas esperadas:", len(verdad))
print(json.dumps(verdad[0]["cabecera"], ensure_ascii=False, indent=2))
print(json.dumps(verdad[0]["tablas"]["referencias"][0], ensure_ascii=False, indent=2))

In [ ]:
# La verificación automática.
correr(f"{PY} -m tests.verificar_extractor data/entrada/lote17.pdf")

### Qué pasa cuando algo se rompe

En vez de tocar el código, le metemos un error a una ficha y miramos qué reporta
el verificador. Esto es lo que **no** ves mirando columnas a ojo.

In [ ]:
from tests.verificar_extractor import _diferencias
import copy

buena = verdad[0]
rota = copy.deepcopy(buena)
rota["tablas"]["referencias"][0]["Antigüedad"] = "12/5/06"      # año de 2 dígitos
del rota["tablas"]["referencias"][1]                            # se come una fila

for d in _diferencias(buena, rota):
    print(" -", d)

## 4. El dominio: reglas sucias

El 80% del trabajo real es que **la realidad no viene tipada**.

Mirá la columna Antigüedad del cliente 2: diez formas distintas de escribir lo
mismo, en la misma columna.

In [ ]:
from datetime import date
from pipeline.dominio.normalizadores import parse_antiguedad, parse_plazo, cuit_valido

EMISION = date(2026, 7, 3)   # la fecha de emisión del PDF, NO date.today()

filas = ["| crudo | interpretado | por qué |", "|---|---|---|"]
casos = [
    ("12/5/2006", "fecha completa"),
    ("05/2019",   "mes/año → día 01"),
    ("05/18",     "año de 2 dígitos → pivote en 26"),
    ("2011",      "4 dígitos = año → 01/07"),
    ("2 meses",   "relativo a la EMISIÓN, no a hoy"),
    ("1 año",     "relativo a la EMISIÓN"),
    ("3",         "1-3 dígitos = cantidad de años"),
    ("reciente",  "palabra sola → sin fecha"),
    ("-",         "vacío"),
    ("n/c",       "no consta"),
]
for crudo, porque in casos:
    filas.append(f"| `{crudo}` | `{parse_antiguedad(crudo, EMISION)}` | {porque} |")
display(Markdown("\n".join(filas)))

**La decisión de diseño más importante del proyecto está en esa tabla.**

`2 meses` — ¿dos meses desde cuándo? Si lo anclás a `date.today()`, el mismo PDF
da resultados distintos según el día que lo corras. Se ancla a la **fecha de
emisión**: el PDF es una función pura.

In [ ]:
# Plazo: válido de 0 a 90 (son cuotas). Lo que no entra, se marca.
for crudo in ["30", "90", "3060", "150", "-"]:
    valor, problema = parse_plazo(crudo)
    print(f"  {crudo!r:>8} -> valor={valor!r:<6} problema={problema}")

print()
# CUIT inválido NO bloquea: se carga y se marca. Un cliente nunca desaparece.
for c in ["30-70920459-5", "30-70920459-4"]:
    print(f"  {c} -> válido: {cuit_valido(c)}")

## 5. El extractor de PDF

Las mañas del formato, y por qué **no** contaminan el dominio.

In [ ]:
import pdfplumber
pdf = pdfplumber.open("data/entrada/lote17.pdf")

# TRAMPA 1: el cliente 9 se derrama a la página 10, que NO repite el
# encabezado del cliente. Mirada de a una página, es indistinguible de un
# cliente nuevo.
print("página 9 (arranca el cliente):")
for l in pdf.pages[8].extract_text_lines()[:2]:
    print("   ", l["text"][:70])

print("\npágina 10 (continuación, sin encabezado de cliente):")
for l in pdf.pages[9].extract_text_lines()[:2]:
    print("   ", l["text"][:70])

In [ ]:
# TRAMPA 2: las filas de subgrupo no son datos, son títulos.
# Una celda combinada que abarca las 12 columnas.
fila = pdf.pages[0].find_tables()[0].extract()[1]
print("fila 1 de la tabla:")
print("  ", fila[:3], "...")
print()
print("una sola celda con texto  =>  es un TÍTULO, no una fila de datos")
print("hay que bajarlo a las filas que le siguen: la solicitud es parte de la clave")

In [ ]:
# TRAMPA 3: qué tabla es cuál. No por posición ni por el título de sección
# (la página de continuación no lo repite): por SUS PROPIOS encabezados.
from pipeline.extractores.comun import seccion_de

for p in (0, 9):
    for t in pdf.pages[p].find_tables():
        enc = [(c or "").replace("\n", " ") for c in t.extract()[0]]
        print(f"  pág {p+1}: {seccion_de(enc):<12} <- encabezados {enc[:2]}")
pdf.close()

## 6. Enchufar DOCX

Acá se cobra el contrato de la sección 2.

Mismos datos, otro formato, **el mismo ground truth**. Si los dos extractores
cumplen el contrato, tienen que producir exactamente lo mismo.

In [ ]:
correr(f"{PY} demo/generar_docx_fake.py | head -3")
correr(f"{PY} -m tests.verificar_extractor data/entrada/lote17.docx | tail -2")

In [ ]:
# El contraste entre librerías: la MISMA celda combinada, entregada distinto.
from docx import Document
doc = Document("data/entrada/lote17.docx")
celdas = [c.text for c in doc.tables[0].rows[1].cells]
print("python-docx la devuelve REPETIDA en las 12 columnas:")
print("   cantidad de celdas:", len(celdas), "| valores distintos:", len(set(celdas)))
print()
print("pdfplumber la devuelve como texto en la 1ra celda y None en las otras 11.")
print("Distinta maña, misma detección de fondo: una sola celda con texto = título.")

In [ ]:
# LA PRUEBA: extraer los dos y comparar. Idénticos salvo 'origen'.
cli("extraer data/entrada/lote17.pdf  -o /tmp/a.jsonl", mostrar=False)
cli("extraer data/entrada/lote17.docx -o /tmp/b.jsonl", mostrar=False)

import json
sin_origen = lambda p: [{k: v for k, v in json.loads(l).items() if k != "origen"}
                        for l in open(p, encoding="utf-8")]
print("JSONL desde PDF == JSONL desde DOCX:", sin_origen("/tmp/a.jsonl") == sin_origen("/tmp/b.jsonl"))

## 7. Revisión humana: el Excel

Al humano no se lo saca del loop: se lo **reubica**. En vez de copiar 300 fichas,
mira una planilla y decide.

Lo primero que hay que leer es la **reconciliación**: los números tienen que
cerrar contra el documento. *"No hubo excepciones"* no alcanza.

In [ ]:
cli("extraer data/entrada/lote17.pdf")

In [ ]:
cli("revisar data/salida/lote17.jsonl", mostrar=False)

# La hoja Problemas es el producto: si está vacía, cargás tranquilo.
from openpyxl import load_workbook
wb = load_workbook("data/salida/lote17.xlsx")
print("hojas:", wb.sheetnames, "\n")

filas = ["| " + " | ".join(str(c) for c in f) + " |"
         for f in wb["Problemas"].iter_rows(values_only=True)]
filas.insert(1, "|---|---|---|---|")
display(Markdown("\n".join(filas)))

**Ojo con la dirección de la flecha.** El JSONL es la fuente de verdad; el Excel
es una foto. Las correcciones NO vuelven editando el Excel: se corrige la regla
en el código y se re-corre. Si el Excel fuera editable de vuelta, a la segunda
corrida nadie sabría cuál de los dos tiene razón.

## 8. La web falsa

Es una web de verdad: Flask + SQLite, con buscador, tabla y formulario. Está en
el repo para que el curso corra entero sin credenciales ni sistemas externos.

Como en Colab no hay pantalla, la vemos con **capturas** que saca Playwright.

In [ ]:
# Se siembra con las filas ya existentes pero VACÍAS: esa es la situación real.
correr(f"{PY} web_demo/sembrar.py data/salida/lote17.jsonl")

servidor = subprocess.Popen([PY, "web_demo/app.py"],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
print("\nservidor levantado:", servidor.poll() is None)

In [ ]:
def foto(url, alto=700, archivo="/tmp/shot.png"):
    """Captura la web y la muestra acá.

    Se llama a demo/captura.py como proceso aparte a propósito: la API
    sincrónica de Playwright choca con el event loop que Jupyter ya tiene
    corriendo. En un proceso propio no hay conflicto.
    """
    subprocess.run([PY, "demo/captura.py", url, archivo, "--alto", str(alto)],
                   check=True, capture_output=True)
    return Image(filename=archivo)

BASE = "http://127.0.0.1:5000"
CUIT = "33645428418"     # SERVICIOS CCI SRL: tiene solicitudes Y alertas

foto(f"{BASE}/admin/buscar?cuit={CUIT}")

In [ ]:
# El formulario de destino. Fijate en el bloque rojo de abajo.
foto(f"{BASE}/admin/solicitudes/1/financieras/1/edit", alto=850)

**El problema del id.** La URL es
`/admin/solicitudes/1/financieras/1/edit` — y ninguno de esos dos números está
en el PDF. No se pueden calcular.

Hay que buscar por CUIT, entrar a la solicitud y **cosechar el href del lápiz**.
Es más lento, y es a propósito: un id inventado escribe en la fila de otro.

**Y los campos de abajo (Monto Asegurado, Seguro, Doc) no hay que tocarlos
nunca.** Están ahí para que la lista blanca sea una defensa verificable y no un
comentario.

## 9. La carga

Los selectores y las URLs son **datos**, no código: viven en
`config/mapeo_web.yaml`. Apuntar el pipeline a otra web es editar ese archivo.

In [ ]:
print(Path("config/mapeo_web.yaml").read_text(encoding="utf-8")[:1200])

In [ ]:
# SIEMPRE dry-run primero: llena el formulario y NO guarda.
cli("cargar data/salida/lote17.jsonl --dry-run --limite 3")

### Las tres defensas

1. **Lista blanca** — sólo se escriben los campos declarados. Uno fuera de la
   lista **aborta la fila** sin tocar la web.
2. **Nada más se movió** — se fotografían *todos* los inputs antes y después.
3. **Read-back** — se reabre el formulario y se compara. Que el submit no
   explote no significa que el dato entró.

In [ ]:
# Defensa 1 en vivo: intentamos escribir un campo que no está en la lista blanca.
from pipeline.carga.mapeo import Mapeo
from pipeline.carga.navegador import CargadorNulo
from pipeline.contratos import ItemDeCarga

mapeo = Mapeo.cargar("config/mapeo_web.yaml")
malo = ItemDeCarga(formulario="referencias", clave="demo",
                   busqueda={"cuit": "x"},
                   campos={"condicion": "Contado", "monto_asegurado": "999"})

with CargadorNulo(mapeo) as c:
    print(c.cargar(malo).detalle)

In [ ]:
# La carga real. Tarda: son 80 filas y cada una navega de verdad.
cli("cargar data/salida/lote17.jsonl | tail -4")

In [ ]:
# Cómo quedó la web. Y la póliza, intacta.
import sqlite3
con = sqlite3.connect("web_demo/datos.db")
q = lambda s: con.execute(s).fetchone()[0]
print("referencias cargadas :", q("SELECT COUNT(*) FROM financieras WHERE condicion != ''"),
      "de", q("SELECT COUNT(*) FROM financieras"))
print("alertas cargadas     :", q("SELECT COUNT(*) FROM alertas WHERE tipo != ''"),
      "de", q("SELECT COUNT(*) FROM alertas"))
print("pólizas pisadas      :", q("SELECT COUNT(*) FROM financieras WHERE monto_asegurado = ''"), "(esperado 0)")
print("expedientes pisados  :", q("SELECT COUNT(*) FROM alertas WHERE expediente = ''"), "(esperado 0)")
con.close()

foto(f"{BASE}/admin/solicitudes/1")

### Verlo en vivo, con el navegador de verdad

Las capturas de arriba son la única forma de ver la web **dentro de Colab**,
porque la máquina de Colab no tiene pantalla.

Pero si estás corriendo esto **en tu máquina**, hay algo mucho mejor: podés
abrir el navegador de verdad y mirar cómo se llenan los campos, uno por uno, en
tiempo real. Es la mejor demo del curso y no entra en un cuaderno.

En una terminal, con el venv activado:

```bash
# dry-run: llena el formulario y NO guarda. Ideal para mostrar sin ensuciar.
python -m pipeline.cli cargar data/salida/lote17.jsonl --dry-run --limite 3 --ver --lento 400

# la carga real, mirándola
python -m pipeline.cli cargar data/salida/lote17.jsonl --limite 5 --ver --lento 400
```

- `--ver` abre Chromium con ventana en vez de headless.
- `--lento 400` mete 400 ms entre acción y acción. Sin eso Playwright va
  demasiado rápido para que se entienda lo que está pasando.
- `--limite N` para no esperar las 80 filas en cámara.

**Ese es el momento del curso donde se ve que esto no es magia:** es un
navegador haciendo, muy rápido, exactamente lo mismo que hacía la persona.

## 10. Idempotencia y conflictos

Los dos casos que separan un script de una herramienta.

In [ ]:
# Volver a correr no duplica nada: saltea lo ya cargado.
cli("cargar data/salida/lote17.jsonl | tail -3")

### La trampa del dry-run

*¿Debería `--dry-run` anotar estado?*

**No.** Si anotara, la corrida real saltearía filas que nunca se escribieron y
quedarían vacías para siempre. Es un bug que tuvimos y corregimos.

### El conflicto

El cliente STENFAR tiene **dos filas con el mismo informante y la misma fecha**
en la misma solicitud. En la web las dos matchean igual.

Agarrar "la primera" escribiría en la fila equivocada, y nadie se enteraría
nunca. La decisión: **no se cargan**.

In [ ]:
con = sqlite3.connect("web_demo/datos.db")
print("las dos filas en conflicto, después de la carga:\n")
for f in con.execute("""SELECT fecha, informante, condicion, antiguedad, monto_asegurado
                        FROM financieras WHERE informante = 'MICROGLOBAL'
                        AND fecha = '02/06/2026'"""):
    print("  ", f)
con.close()
print("\ncondicion y antiguedad vacías: el pipeline se negó a adivinar.")

> **Automatizar bien no es cargar todo. Es saber qué no cargar.**

## 11. Adaptarlo a lo tuyo

Tres puntos de reemplazo. Ninguno obliga a tocar el resto.

| Qué cambia | Qué tocás |
|---|---|
| Otro **formato** de entrada | una clase en `pipeline/extractores/` + una línea en el registro |
| Otro **rubro** | `pipeline/dominio/esquema.py` (los normalizadores sobreviven) |
| Otra **web** | `config/mapeo_web.yaml` |
| Otro **formulario** en la misma web | un bloque en el YAML + un `elif` en `_ir_a_la_tabla` |

Lo que construimos no es un cargador de solicitudes financieras: es un esqueleto
con dos enchufes y tres defensas. El caso que usamos es una excusa.

In [ ]:
# Cerrar el servidor al terminar.
servidor.terminate()
print("servidor apagado")

---

### Reset entre tomas

Deja la web como la encuentra el operador: todas las filas existen, todos los
campos en blanco.

```bash
rm -f web_demo/datos.db data/salida/estado_carga.db
python web_demo/sembrar.py data/salida/lote17.jsonl
```